In [1]:
# run autoreload
get_ipython().run_line_magic('load_ext', 'autoreload')
get_ipython().run_line_magic('autoreload', '2')

In [7]:
import json
import sys
import importlib
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

repo_root = Path.cwd()
if not (repo_root / "src" / "zjet_corrections").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str((repo_root / "src").resolve()))

import zjet_corrections.notebook_utils as nbutils
importlib.reload(nbutils)

# -------------------- Persistence --------------------
CONFIG_FILE = Path(".analysis_widget_config.json")

DEFAULTS = {
    "casa": True,
    "test": False,
    "useDefault": False,
    "executor_mode": "futures",
    "mode": "validation",
    "era": "2016",
    "dataset": "pythia",
    "chunksize": 50000,
    "chunksize_test": 200000,
    "group_mode": "all_in_one",
    "prependstr": "root://xcache/",
    "systematic_profile": "all_syst",
}


def load_config():
    """Load last saved config if available, otherwise return defaults."""
    if CONFIG_FILE.exists():
        try:
            with open(CONFIG_FILE, "r") as f:
                loaded = json.load(f)

            config = DEFAULTS.copy()
            config.update(loaded)
            return config

        except Exception as e:
            print(f"Warning: could not read {CONFIG_FILE}: {e}")
            print("Falling back to defaults.")
    return DEFAULTS.copy()



def save_config(config):
    """Save current config to disk."""
    try:
        with open(CONFIG_FILE, "w") as f:
            json.dump(config, f, indent=2)
    except Exception as e:
        print(f"Warning: could not save config: {e}")



def apply_config_to_widgets(config):
    """Push config values into widgets."""
    casa_w.value = config["casa"]
    test_w.value = config["test"]
    useDefault_w.value = config["useDefault"]
    executor_mode_w.value = config.get("executor_mode", DEFAULTS["executor_mode"])
    mode_w.value = config["mode"]
    era_w.value = config["era"]
    dataset_w.value = config["dataset"]
    chunksize_w.value = config["chunksize"]
    chunksize_test_w.value = config["chunksize_test"]
    group_mode_w.value = config["group_mode"]
    prependstr_w.value = config["prependstr"]
    systematic_profile_w.value = config["systematic_profile"]



def get_config_from_widgets():
    """Pull current widget values into a config dict."""
    return {
        "casa": casa_w.value,
        "test": test_w.value,
        "useDefault": useDefault_w.value,
        "executor_mode": executor_mode_w.value,
        "mode": mode_w.value,
        "era": era_w.value,
        "dataset": dataset_w.value,
        "chunksize": chunksize_w.value,
        "chunksize_test": chunksize_test_w.value,
        "group_mode": group_mode_w.value,
        "prependstr": prependstr_w.value,
        "systematic_profile": systematic_profile_w.value,
    }



def apply_widget_values(save=False, show_output=True):
    """
    Read current widget values and apply them to notebook variables.
    Optionally save them and/or print them.
    """
    global casa, test, mode, era, data, dataset
    global chunksize, chunksize_test, useDefault, executor_mode
    global group_mode, prependstr, systematic_profile
    global systematics_list, jet_systematics_list

    casa = casa_w.value
    test = test_w.value
    mode = mode_w.value
    era = era_w.value
    dataset = dataset_w.value
    data = dataset == "data"
    chunksize = chunksize_w.value
    chunksize_test = chunksize_test_w.value
    useDefault = useDefault_w.value
    executor_mode = executor_mode_w.value
    group_mode = group_mode_w.value
    prependstr = prependstr_w.value
    systematic_profile = systematic_profile_w.value
    systematics_list, jet_systematics_list = nbutils.resolve_systematics(systematic_profile)

    if save:
        save_config(get_config_from_widgets())

    if show_output:
        with output:
            output.clear_output()
            print("Applied settings:")
            print(f"casa = {casa}")
            print(f"test = {test}")
            print(f"mode = {mode}")
            print(f"era = {era}")
            print(f"data = {data}")
            print(f"dataset = {dataset}")
            print(f"systematic_profile = {systematic_profile}")
            print(f"systematics_list = {systematics_list}")
            print(f"jet_systematics_list = {jet_systematics_list}")
            print(f"chunksize = {chunksize}")
            print(f"chunksize_test = {chunksize_test}")
            print(f"useDefault = {useDefault}")
            print(f"executor_mode = {executor_mode}")
            print(f"group_mode = {group_mode}")
            print(f"prependstr = {prependstr}")
            print("output_dir = outputs/")
            if save:
                print(f"\nSaved to: {CONFIG_FILE}")


# Load last saved config (or defaults if none exists)
CONFIG = load_config()

DATASET_OPTIONS = nbutils.DATASET_OPTIONS
ERA_OPTIONS = nbutils.ERA_OPTIONS
MODE_OPTIONS = nbutils.MODE_OPTIONS
SYSTEMATIC_PROFILE_OPTIONS = nbutils.SYSTEMATIC_PROFILE_OPTIONS
EXECUTOR_MODE_OPTIONS = nbutils.EXECUTOR_MODE_OPTIONS

if CONFIG["dataset"] not in DATASET_OPTIONS:
    CONFIG["dataset"] = DEFAULTS["dataset"]
if CONFIG["era"] not in ERA_OPTIONS:
    CONFIG["era"] = DEFAULTS["era"]
if CONFIG["mode"] not in MODE_OPTIONS:
    CONFIG["mode"] = DEFAULTS["mode"]
if CONFIG.get("systematic_profile") not in SYSTEMATIC_PROFILE_OPTIONS:
    CONFIG["systematic_profile"] = DEFAULTS["systematic_profile"]
if CONFIG.get("executor_mode") not in EXECUTOR_MODE_OPTIONS:
    CONFIG["executor_mode"] = DEFAULTS["executor_mode"]

# -------------------- Widgets --------------------
casa_w = widgets.Checkbox(value=CONFIG["casa"], description="casa")
test_w = widgets.Checkbox(value=CONFIG["test"], description="test")
useDefault_w = widgets.Checkbox(value=CONFIG["useDefault"], description="useDefault")
executor_mode_w = widgets.Dropdown(
    options=EXECUTOR_MODE_OPTIONS,
    value=CONFIG["executor_mode"],
    description="executor"
)

mode_w = widgets.Dropdown(
    options=MODE_OPTIONS,
    value=CONFIG["mode"],
    description="mode"
)

era_w = widgets.Dropdown(
    options=ERA_OPTIONS,
    value=CONFIG["era"],
    description="era"
)

dataset_w = widgets.Dropdown(
    options=DATASET_OPTIONS,
    value=CONFIG["dataset"],
    description="dataset"
)

systematic_profile_w = widgets.Dropdown(
    options=SYSTEMATIC_PROFILE_OPTIONS,
    value=CONFIG["systematic_profile"],
    description="syst profile"
)

chunksize_w = widgets.IntText(
    value=CONFIG["chunksize"],
    description="chunksize"
)

chunksize_test_w = widgets.IntText(
    value=CONFIG["chunksize_test"],
    description="test chunk"
)

group_mode_w = widgets.Dropdown(
    options=["all_in_one", "per_group"],
    value=CONFIG["group_mode"],
    description="group_mode"
)

prependstr_w = widgets.Text(
    value=CONFIG["prependstr"],
    description="prependstr"
)

run_button = widgets.Button(
    description="Apply settings",
    button_style="success"
)

reset_button = widgets.Button(
    description="Reset to defaults",
    button_style="warning"
)

output = widgets.Output()

# -------------------- Layout --------------------
ui = widgets.VBox([
    widgets.HTML("<h3>Analysis configuration</h3>"),
    widgets.HBox([casa_w, test_w, useDefault_w]),
    executor_mode_w,
    mode_w,
    era_w,
    dataset_w,
    systematic_profile_w,
    chunksize_w,
    chunksize_test_w,
    group_mode_w,
    prependstr_w,
    widgets.HBox([run_button, reset_button]),
    output
])

display(ui)

# -------------------- Button actions --------------------
def on_run_clicked(b):
    apply_widget_values(save=True, show_output=True)



def on_reset_clicked(b):
    apply_config_to_widgets(DEFAULTS)
    apply_widget_values(save=True, show_output=True)

    with output:
        output.clear_output()
        print("Reset all widget values to defaults and applied them.")
        print(f"Saved defaults to: {CONFIG_FILE}")
        print(f"casa = {casa}")
        print(f"test = {test}")
        print(f"mode = {mode}")
        print(f"era = {era}")
        print(f"data = {data}")
        print(f"dataset = {dataset}")
        print(f"systematic_profile = {systematic_profile}")
        print(f"systematics_list = {systematics_list}")
        print(f"jet_systematics_list = {jet_systematics_list}")
        print(f"chunksize = {chunksize}")
        print(f"chunksize_test = {chunksize_test}")
        print(f"useDefault = {useDefault}")
        print(f"executor_mode = {executor_mode}")
        print(f"group_mode = {group_mode}")
        print(f"prependstr = {prependstr}")
        print("output_dir = outputs/")


run_button.on_click(on_run_clicked)
reset_button.on_click(on_reset_clicked)

# -------------------- Auto-apply on cell execution --------------------
apply_widget_values(save=False, show_output=True)


In [3]:
paths = nbutils.get_analysis_paths(repo_root)
HT_BINS = nbutils.HT_BINS

SAMPLES_DATA_DIR = paths.samples_data_dir
SAMPLES_MC_DIR = paths.samples_mc_dir
SAMPLES_BKG_DIR = paths.samples_bkg_dir
SAMPLES_MC_LOCAL_DIR = paths.samples_mc_local_dir

samplePath = nbutils.SamplePath(era)

# In[4]:  -------------------- Imports (do once) --------------------
import os
import time
import pickle
import importlib

import numpy as np
import awkward as ak
import uproot

import coffea
from coffea.nanoevents import NanoAODSchema
from coffea import processor
from IPython.display import Audio

importlib.reload(nbutils)
import dask
dask.config.set({
    "distributed.logging.distributed": "error",
    "distributed.logging.bokeh": "error",
    "distributed.logging.tornado": "error",
})


In [4]:
# In[5]:  -------------------- Helpers --------------------
NanoAODSchema.warn_missing_crossrefs = False

format_time = nbutils.format_time
iter_groups = nbutils.iter_groups
build_fileset_from_txts = nbutils.build_fileset_from_txts
build_backgrounds_fileset = nbutils.build_backgrounds_fileset
build_local_pythia_fileset = nbutils.build_local_pythia_fileset
make_runner = nbutils.make_runner
ensure_client = nbutils.ensure_client
upload_package_if_casa = nbutils.upload_package_if_casa
run_once = nbutils.run_once
save_output = nbutils.save_output
make_output_filename = nbutils.make_output_filename
get_group_tag = nbutils.get_group_tag
ST_FILES = nbutils.ST_FILES


In [5]:
# In[6]:  -------------------- Create client --------------------
client = ensure_client(casa=casa, test=test, useDefault = useDefault, executor_mode=executor_mode)
upload_package_if_casa(client, casa=casa)

Running locally with 1-2 files (test=True)
Using FuturesExecutor without a Dask client.


In [6]:
# In[7]:  -------------------- Build fileset(s), run, and save immediately --------------------
outs = []  # keep multiple outputs if you run multiple groups



def run_save_append(
    fileset,
    i,
    *,
    client=None,
    test=False,
    data=False,
    chunksize=None,
    chunksize_test=None,
):
    out_i = run_once(
        fileset,
        client=client,
        test=test,
        data=data,
        mode=mode,
        systematic_profile=systematic_profile,
        chunksize=chunksize,
        chunksize_test=chunksize_test,
        executor_mode=executor_mode,
    )

    tag = get_group_tag(i, era, group_mode)
    fout = make_output_filename(
        data=data,
        dataset=dataset,
        tag=tag,
        mode=mode,
        test=test,
    )

    save_output(out_i, fout)
    print(f"[{i+1}] Saved: {fout}")

    outs.append(out_i)
    return out_i


if data:
    for i, group in enumerate(iter_groups(samplePath.data, group_mode)):
        fileset = build_fileset_from_txts(
            group,
            SAMPLES_DATA_DIR,
            prependstr,
            split_ht=False,
        )
        run_save_append(
            fileset,
            i,
            client=client,
            test=test,
            data=True,
            chunksize=chunksize,
            chunksize_test=chunksize_test,
        )

else:
    if dataset == "pythia":
        for i, group in enumerate(iter_groups(samplePath.pythia, group_mode)):
            fileset = build_fileset_from_txts(
                group,
                SAMPLES_MC_DIR,
                prependstr,
                split_ht=True,
                ht_bins=HT_BINS,
            )
            run_save_append(
                fileset,
                i,
                client=client,
                test=test,
                data=False,
                chunksize=chunksize,
                chunksize_test=chunksize_test,
            )

    elif dataset == "pythia_local":
        fileset = build_local_pythia_fileset(SAMPLES_MC_LOCAL_DIR, era)
        run_save_append(
            fileset,
            0,
            client=client,
            test=test,
            data=False,
            chunksize=chunksize,
            chunksize_test=chunksize_test,
        )

    elif dataset == "pythia2":
        fileset = build_fileset_from_txts(
            ["inclusive_UL16NanoAODv9.txt"],
            SAMPLES_MC_DIR,
            prependstr,
            split_ht=False,
        )
        run_save_append(
            fileset,
            0,
            client=client,
            test=test,
            data=False,
            chunksize=chunksize,
            chunksize_test=chunksize_test,
        )

    elif dataset == "herwig":
        for i, group in enumerate(iter_groups(samplePath.herwig, group_mode)):
            fileset = build_fileset_from_txts(
                group,
                SAMPLES_MC_DIR,
                prependstr,
                split_ht=False,
            )
            run_save_append(
                fileset,
                i,
                client=client,
                test=test,
                data=False,
                chunksize=chunksize,
                chunksize_test=chunksize_test,
            )

    elif dataset == "powheg":
        fileset = build_fileset_from_txts(
            ["powheg_UL18NanoAODv9_inclusive.txt"],
            SAMPLES_MC_DIR,
            prependstr,
            split_ht=False,
        )
        run_save_append(
            fileset,
            0,
            client=client,
            test=test,
            data=False,
            chunksize=chunksize,
            chunksize_test=chunksize_test,
        )

    elif dataset == "st":
        fileset = build_fileset_from_txts(
            ST_FILES,
            SAMPLES_MC_DIR,
            prependstr,
            split_ht=False,
        )
        run_save_append(
            fileset,
            0,
            client=client,
            test=test,
            data=False,
            chunksize=chunksize,
            chunksize_test=chunksize_test,
        )

    elif dataset == "backgrounds":
        fileset = build_backgrounds_fileset(
            SAMPLES_BKG_DIR,
            prependstr,
        )
        run_save_append(
            fileset,
            0,
            client=client,
            test=test,
            data=False,
            chunksize=chunksize,
            chunksize_test=chunksize_test,
        )

    else:
        print(f"Dataset is {dataset} and it is not in the list")


# In[8]:  -------------------- Choose what to keep in `out` --------------------
out = outs[-1] if outs else None


# In[10]:  -------------------- Analysis / plotting zone --------------------
# Keep plotting down here so the run block stays clean.
# (Your existing plotting cells can remain, just moved below this line.)

print(f"Number of group outputs: {len(outs)}")

if client is not None:
    client.close()


Running over: ['pythia_UL18NanoAODv9_HT-400to600'] 
Running over test files: ['pythia_UL18NanoAODv9_HT-400to600']
[DEBUG] Registered Histograms dict_keys(['ptjet_mjet_u_reco', 'ptjet_mjet_g_reco', 'response_matrix_u', 'response_matrix_g', 'ptjet_mjet_u_gen', 'ptjet_mjet_g_gen', 'ptz_mz_reco', 'sumw', 'nev', 'cutflow'])


Output()

Output()

/home/aritra/.pyenv/versions/coffea_latest/lib/python3.11/site-packages/coffea/nanoevents/schemas/nanoaod.py:283: RuntimeWarning: Missing cross-reference index for LowPtElectron_electronIdx => Electron
  warnings.warn(
/home/aritra/.pyenv/versions/coffea_latest/lib/python3.11/site-packages/coffea/nanoevents/schemas/nanoaod.py:283: RuntimeWarning: Missing cross-reference index for LowPtElectron_photonIdx => Photon
  warnings.warn(
/home/aritra/.pyenv/versions/coffea_latest/lib/python3.11/site-packages/coffea/nanoevents/schemas/nanoaod.py:322: RuntimeWarning: Branch Photon_mass already exists but its values will be replaced with 0.0
  warnings.warn(
/home/aritra/.pyenv/versions/coffea_latest/lib/python3.11/site-packages/coffea/nanoevents/schemas/nanoaod.py:322: RuntimeWarning: Branch Photon_charge already exists but its values will be replaced with 0.0
  warnings.warn(


[DEBUG] Systematics ['nominal']
[DEBUG] Current Mode reweight_pythia
[INFO] Starting processing for dataset: pythia_UL18NanoAODv9_HT-400to600 and file: /mnt/8A04C21E04C20CDF/wsLinux/zjet_corrections/samples_mc/files/store/mc/RunIISummer20UL18NanoAODv9/DYJetsToLL_M-50_HT-400to600_TuneCP5_PSweights_13TeV-madgraphMLM-pythia8/NANOAODSIM/106X_upgrade2018_realistic_v16_L1v1-v1/270000/90EAC274-6CB4-BB42-8F60-BF38CBBC26DC.root
[DEBUG] Total events in chunk: 32720
[DEBUG] Jackknife resampling not enabled, processing all events together.
[INFO] year: RunIISummer20UL18NanoAODv9, ht_bin: HT-400to600, herwig: False
[DEBUG] Weights initialized
ht bin HT-400to600
[INFO] Applying MET filters
[DEBUG] PU weights (nom, up, down) : [0.99084239 1.00662441 0.88210136 1.02818737 0.91235547 1.01166006
 1.03718    1.07108805 0.99286314 0.98016382]
[DEBUG] pdf weights (nom, up, down) : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[DEBUG] L1 prefiring weights (nom, up, down) : [1.        0.9947555 1.        1.        0.996101

...

[DEBUG] Jet Corrections Applied
[DEBUG] Available Jet systematics ['nominal']
[DEBUG] Processing jet systematic: nominal
[DEBUG] FatJet pt before correction: [[240], [214], [297, 189], [246, 196], ..., [...], [346], [339, 306], [258]]
How many none in Fatjet.mass before processing inside jmrnom 4418
Genmass inside jmrsf [[], [43.1], [40.7], [55.8, 33.3], ..., [48.1], [46.9], [109, 59.7], [45.9]]
How many none in Fatjet.matched_gen inside jmrnom 0
[[], [0.99], [0.921], [1.01, 1.03], ..., [0.996], [1.01], [1.01, 1.01], [0.979]]
How many none in Fatjet inside jmrnom 0
How many none in Fatjet.mass that is being returned inside jmrnom 4421
[DEBUG] FatJet pt after correction: [[240], [213], [299, 189], [248], [...], ..., [250], [351], [339, 306], [259]]
[DEBUG] Sum of this sel is 3196
[DEBUG] Len? 3196  
[DEBUG] Padded Electron/Muon collections to minimum length 2 per event
[INFO] Lepton weights added
[DEBUG] Total reco events passing all selection: 2470
[DEBUG] Total gen events passing all 

/home/aritra/.pyenv/versions/coffea_latest/lib/python3.11/site-packages/awkward/_nplikes/array_module.py:289: RuntimeWarning: invalid value encountered in log10
  return impl(*broadcasted_args, **(kwargs or {}))
/home/aritra/.pyenv/versions/coffea_latest/lib/python3.11/site-packages/awkward/_nplikes/array_module.py:289: RuntimeWarning: divide by zero encountered in log10
  return impl(*broadcasted_args, **(kwargs or {}))


[DEBUG] herwig WEights both g? [1.09, 1.23, 1.24, 1.12, 0.649, 1.11, ..., 0.638, 1.05, 1.13, 1.02, 0.927, 0.91]
Fatjet y  [-0.527, 1.44, 1.36, 0.402, 1.01, ..., 0.637, -0.912, 0.166, -1.51, -2.03]
Fatjet eta  [-0.529, 1.49, 1.37, 0.407, 1.06, 0.59, ..., 0.654, -0.92, 0.167, -1.51, -2.03]
[DEBUG] Len of ptreco 908 mreco 908 syst nominal channel ee dataset pythia_UL18NanoAODv9_HT-400to600
[DEBUG] ptreco sample [536, 206, 524, 241, 315, 212, 342, 247, 209, 264]
[DEBUG] mreco sample [54, 69.3, 53.5, 41.5, 118, 58.6, 122, 42.8, 59.5, 74.3]
[DEBUG] mreco_g sample [5.87, 70.6, 28, 37.8, 102, 49.2, 126, 13.2, 26.8, 71.8]


[INFO] Scaled ptjet_mjet_u_reco for dataset pythia_UL18NanoAODv9_HT-400to600 by 10.688575 = 5.174 * 59830.0 / 32720.0
[INFO] Scaled ptjet_mjet_g_reco for dataset pythia_UL18NanoAODv9_HT-400to600 by 10.688575 = 5.174 * 59830.0 / 32720.0
[INFO] Scaled response_matrix_u for dataset pythia_UL18NanoAODv9_HT-400to600 by 10.688575 = 5.174 * 59830.0 / 32720.0
[INFO] Scaled response_matrix_g for dataset pythia_UL18NanoAODv9_HT-400to600 by 10.688575 = 5.174 * 59830.0 / 32720.0
[INFO] Scaled ptjet_mjet_u_gen for dataset pythia_UL18NanoAODv9_HT-400to600 by 10.688575 = 5.174 * 59830.0 / 32720.0
[INFO] Scaled ptjet_mjet_g_gen for dataset pythia_UL18NanoAODv9_HT-400to600 by 10.688575 = 5.174 * 59830.0 / 32720.0
[INFO] Scaled ptz_mz_reco for dataset pythia_UL18NanoAODv9_HT-400to600 by 10.688575 = 5.174 * 59830.0 / 32720.0
Done. time taken 31s
Output written to /mnt/8A04C21E04C20CDF/wsLinux/zjet_corrections/outputs/mass_reweight_pythia_local_2018_TEST.pkl with size 1.9 MB
[1] Saved: /mnt/8A04C21E04C20C

/mnt/8A04C21E04C20CDF/wsLinux/zjet_corrections/src/zjet_corrections/hist_utils.py:23: UserWarning: Please use 'Weight()' instead of 'Weight'
  hnew = hist.Hist(
/home/aritra/.pyenv/versions/coffea_latest/lib/python3.11/site-packages/hist/basehist.py:549: UserWarning: List indexing selection is experimental. Removed bins are not placed in overflow.
  return super().__getitem__(self._index_transform(index))


In [13]:
fileset

{'pythia_UL18NanoAODv9_HT-400to600': ['/mnt/8A04C21E04C20CDF/wsLinux/zjet_corrections/samples_mc/files/store/mc/RunIISummer20UL18NanoAODv9/DYJetsToLL_M-50_HT-400to600_TuneCP5_PSweights_13TeV-madgraphMLM-pythia8/NANOAODSIM/106X_upgrade2018_realistic_v16_L1v1-v1/270000/90EAC274-6CB4-BB42-8F60-BF38CBBC26DC.root']}

In [ ]:
for key in out:
    print(key)

In [ ]:
out['ptjet_mjet_g_reco']